# LogFormer full training on Kaggle

Notebook này chuẩn bị embedding và huấn luyện LogFormer trên full HDFS_v1/BGL.

Giao thức dùng chung: HDFS block-level stratified 80/10/10; BGL chronological 200-record windows 80/10/10. Ngưỡng được chọn trên validation, test chỉ dùng đánh giá cuối.

Mặc định: 5 seed (42, 123, 456, 789, 2024), 20 epoch, BGL window 200 records, batch 64, lr 1e-5.
Full training có thể vượt thời gian Kaggle; có thể đổi SEEDS hoặc EPOCHS.

In [ ]:
from pathlib import Path
import os, json, sys, platform, subprocess
import torch

SEEDS = [42, 123, 456, 789, 2024]
EPOCHS = 20
HDFS_SEQUENCE_LENGTH = 20
BGL_WINDOW_SIZE = 200
BATCH_SIZE = 64
LEARNING_RATE = 1e-5

WORKING = Path('/kaggle/working')
WORKING.mkdir(parents=True, exist_ok=True)
PROTOCOL_DIR = WORKING / 'data/splits/benchmark_protocol_v1'
print('Platform:', platform.platform())
print('CUDA:', torch.cuda.is_available())
print('SEEDS:', SEEDS)
print('EPOCHS:', EPOCHS)
print('Working:', WORKING)


## 1. Clone repository và tìm dữ liệu trong Kaggle

Notebook tự clone code mới nhất từ GitHub. Add HDFS_v1 và BGL datasets vào Kaggle.

In [ ]:
REPO_URL = 'https://github.com/rubyhcm/code-papers-doing-copilot.git'
REPO_DIR = Path('/kaggle/working/code-papers-doing-copilot')
if REPO_DIR.exists():
    if not (REPO_DIR / '.git').exists():
        raise FileExistsError(f'Repository path exists but is not a git clone: {REPO_DIR}')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)

repo_root = REPO_DIR
os.chdir(repo_root)
sys.path.insert(0, str(repo_root))
print('Repository:', repo_root)

In [ ]:
def find_path(filename):
    roots = [Path('/kaggle/input'), Path('/kaggle/working'), Path('.')]
    for root in roots:
        if root.exists():
            matches = list(root.rglob(filename))
            if matches:
                return matches[0]
    raise FileNotFoundError(filename)

prepare_script = repo_root / 'scripts/prepare_logformer_embeddings.py'
protocol_script = repo_root / 'scripts/shared_benchmark_protocol.py'
parse_script = repo_root / 'scripts/parse_logs.py'
train_script = repo_root / 'scripts/train_logformer_smoke.py'
for script in (prepare_script, protocol_script, parse_script, train_script):
    if not script.is_file():
        raise FileNotFoundError(f'Required repository script not found: {script}')

hdfs_raw = find_path('HDFS.log')
hdfs_traces = find_path('Event_traces.csv')
hdfs_matrix = find_path('Event_occurrence_matrix.csv')
hdfs_templates = find_path('HDFS.log_templates.csv')
hdfs_labels = find_path('anomaly_label.csv')
bgl_raw = find_path('BGL.log')

print('repo_root:', repo_root)
print('prepare:', prepare_script)
print('HDFS:', hdfs_raw)
print('BGL:', bgl_raw)

## 2. Cài dependency

In [ ]:
get_ipython().system('pip install -q sentence-transformers scikit-learn')
# Avoid downgrading Kaggle's shared cachetools dependency.
get_ipython().system('pip install -q --no-deps drain3==0.9.11')
print('Kaggle dependencies ready')

## 3. Parse BGL raw

In [ ]:
parsed_dir = WORKING / 'data/parsed'
parsed_dir.mkdir(parents=True, exist_ok=True)
bgl_parsed = parsed_dir / 'bgl_structured.jsonl'

cmd = [sys.executable, str(parse_script),
       '--input', str(bgl_raw),
       '--output', str(bgl_parsed),
       '--dataset', 'bgl',
       '--input-format', 'raw']
subprocess.run(cmd, cwd=repo_root, check=True)
print('BGL parsed:', bgl_parsed)

## 4. Tạo split dùng chung và full HDFS/BGL embedding NPZ

`--max-hdfs-blocks 0` nghĩa full HDFS. Cả LightAD và LogFormer phải dùng cùng manifest này.

Pipeline lưu chuỗi bằng `int32` template ID; embedding 768 chiều được lookup theo batch. Cách này tránh giữ mảng dense khoảng 17.7 GB cho HDFS trong RAM. Sau khi cập nhật notebook/repository, chạy lại cell này để tạo mới toàn bộ NPZ.

In [ ]:
cmd = [sys.executable, str(protocol_script),
       '--hdfs-matrix', str(hdfs_matrix),
       '--bgl-parsed', str(bgl_parsed),
       '--output-dir', str(PROTOCOL_DIR),
       '--bgl-window-size', str(BGL_WINDOW_SIZE)]
subprocess.run(cmd, cwd=repo_root, check=True)
print('Shared protocol:', PROTOCOL_DIR / 'manifest.json')

embed_dir = WORKING / 'LogFormer/preprocessed_data'
cmd = [sys.executable, str(prepare_script),
       '--hdfs-traces', str(hdfs_traces),
       '--hdfs-labels', str(hdfs_labels),
       '--hdfs-templates', str(hdfs_templates),
       '--bgl-jsonl', str(bgl_parsed),
       '--output-dir', str(embed_dir),
       '--protocol-dir', str(PROTOCOL_DIR),
       '--hdfs-sequence-length', str(HDFS_SEQUENCE_LENGTH),
       '--bgl-window-size', str(BGL_WINDOW_SIZE),
       '--max-hdfs-blocks', '0']
subprocess.run(cmd, cwd=repo_root, check=True)
print('Embedding manifest:', embed_dir / 'manifest.json')

In [ ]:
import numpy as np
manifest = json.loads((embed_dir / 'manifest.json').read_text())
print(json.dumps(manifest, indent=2, ensure_ascii=False))
if manifest.get('sequence_representation') != 'template_ids; embeddings are looked up per batch':
    raise ValueError('Expected compact template-ID artifacts; rerun step 4 with the latest repository code.')
expected_samples = {
    'HDFS_training.npz': 460048, 'HDFS_validation.npz': 57506, 'HDFS_testing.npz': 57507,
    'BGL_training.npz': 18991, 'BGL_validation.npz': 2374, 'BGL_testing.npz': 2374,
}
for split, expected_count in expected_samples.items():
    path = embed_dir / split
    with np.load(path, allow_pickle=True) as data:
        x, y = data['x'], data['y']
        if x.ndim != 2 or x.dtype != np.int32 or len(x) != expected_count or len(y) != expected_count:
            raise ValueError(f'Unexpected compact split shape/dtype for {split}: {x.shape}, {x.dtype}, {y.shape}')
        print(split, x.shape, y.shape, x.dtype)

## 5. Huấn luyện full HDFS/BGL

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cpu':
    print('GPU chưa bật. Full training trên CPU rất chậm.')

checkpoint_dir = WORKING / 'LogFormer/checkpoints'
checkpoint_dir.mkdir(parents=True, exist_ok=True)

for dataset in ('HDFS','BGL'):
    for seed in SEEDS:
        cmd = [sys.executable, str(train_script),
               '--dataset', dataset,
               '--epochs', str(EPOCHS),
               '--window-size', str(HDFS_SEQUENCE_LENGTH if dataset == 'HDFS' else BGL_WINDOW_SIZE),
               '--batch-size', str(BATCH_SIZE),
               '--lr', str(LEARNING_RATE),
               '--seed', str(seed),
               '--max-train-samples', '0',
               '--device', device]
        cmd.extend(['--preprocessed-dir', str(embed_dir)])
        cmd.extend(['--output', str(checkpoint_dir / f'logformer_full_{dataset.lower()}_seed{seed}_ep{EPOCHS}.pt')])
        print(f'Running {dataset} seed={seed}')
        subprocess.run(cmd, cwd=repo_root, check=True)

## 6. Gom report và lưu aggregate

In [ ]:
reports = []
for path in sorted(checkpoint_dir.glob(f'logformer_full_*_ep{EPOCHS}.json')):
    reports.append(json.loads(path.read_text()))

aggregate_path = WORKING / 'logformer_full_results_shared_protocol_v1.json'
aggregate_path.write_text(json.dumps({
    'protocol_version': manifest['protocol_version'],
    'results': reports,
}, indent=2, ensure_ascii=False))
print('Saved:', aggregate_path)
for report in reports:
    print(report['dataset'], report['seed'], report['test']['f1'], report['elapsed_seconds'])

## 7. Output

Tải từ Kaggle Output:
- `LogFormer/checkpoints/`
- `LogFormer/preprocessed_data/`
- `logformer_full_results_shared_protocol_v1.json`
